# EduPro — Course Demand & Revenue: EDA and Modeling

End-to-end notebook: load raw data → preprocess → engineer features → EDA → train/evaluate models → feature importance.

All heavy lifting lives in `src/`; this notebook calls those functions so logic isn't duplicated.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('../src'))
import pandas as pd
pd.set_option('display.max_columns', None)

## Phase 1: Load raw data

In [2]:
from data_loader import load_raw_data, data_quality_summary

raw = load_raw_data('../data/EduPro_Online_Platform.xlsx')
for name, df in raw.items():
    print(name, df.shape)
data_quality_summary(raw)

users (3000, 5)
teachers (60, 7)
courses (60, 8)
transactions (10000, 7)


,sheet,n_rows,n_columns,missing_values_total,duplicate_rows,unique_keys,key_is_unique
0,users,3000,5,0,0,3000,True
1,teachers,60,7,0,0,60,True
2,courses,60,8,0,0,60,True
3,transactions,10000,7,0,0,10000,True


## Phase 2: Preprocessing

In [3]:
from preprocessing import preprocess_all

processed = preprocess_all(raw)
for name, df in processed.items():
    print(name, df.shape)

users (3000, 5)
teachers (60, 7)
courses (60, 9)
transactions (10000, 8)


## Phase 3/4/6: Feature engineering — course-level dataset and forecasting panel

In [4]:
from feature_engineering import build_course_level_dataset, add_engineered_features, build_forecasting_dataset

course_df = build_course_level_dataset(processed['courses'], processed['transactions'], processed['teachers'])
course_df = add_engineered_features(course_df)
print(course_df.shape)
course_df.head()

(60, 34)


,CourseID,CourseName,CourseCategory,CourseType,CourseLevel,CoursePrice,CourseDuration,CourseRating,IsFreeCourse,EnrollmentCount,TotalRevenue,AverageTransactionAmount,PaidEnrollmentCount,FirstTransactionDate,LastTransactionDate,FreeEnrollmentCount,RevenuePerEnrollment,EnrollmentTrend,RevenueTrend,FirstEnrollmentYear,FirstEnrollmentMonth,FirstEnrollmentQuarter,ActiveDaysSpan,RecencyDays,AvgTeacherRating,AvgTeacherExperience,DistinctTeacherCount,MaxTeacherRating,PriceBand,DurationBucket,RatingTier,ExperienceBucket,CategoryHistoricalDemand,CategoryHistoricalRevenue
0,CR00001,Python Basics,Programming,Paid,Beginner,472.28,11.00,4.74,False,164,77453.92,472.28,164,2025-01-02,2025-12-26,0,472.28,0,0.0,2025,1,1,358,4,4.191646,16.682927,25,4.97,High,Medium,High,Senior,160.50,0.00
1,CR00002,Java Programming,Programming,Free,Intermediate,0.00,37.70,2.43,True,149,0.00,0.00,0,2025-01-03,2025-12-30,149,0.00,1,0.0,2025,1,1,361,0,4.211409,17.013423,20,4.97,Free,VeryLong,Medium,Senior,164.25,19363.48
2,CR00003,C++ for Beginners,Programming,Free,Beginner,0.00,19.53,3.85,True,173,0.00,0.00,0,2025-01-01,2025-12-30,173,0.00,1,0.0,2025,1,1,363,0,4.079017,16.589595,27,4.97,Free,Long,High,Senior,158.25,19363.48
3,CR00004,Advanced Python,Programming,Free,Beginner,0.00,45.13,2.88,True,154,0.00,0.00,0,2025-01-03,2025-12-20,154,0.00,0,0.0,2025,1,1,351,10,4.097987,15.896104,27,4.97,Free,VeryLong,Medium,Senior,163.00,19363.48
4,CR00005,Full Stack Development,Programming,Free,Beginner,0.00,28.68,1.28,True,166,0.00,0.00,0,2025-01-01,2025-12-30,166,0.00,0,0.0,2025,1,1,363,0,4.058554,15.795181,29,4.97,Free,Long,Low,Senior,160.00,19363.48


In [5]:
forecast_df = build_forecasting_dataset(processed['transactions'], processed['courses'])
print(forecast_df.shape)
forecast_df.head()

(600, 16)


,CourseID,YearMonth,MonthlyEnrollment,MonthlyRevenue,Lag1Enrollment,Lag1Revenue,RollingAvgEnrollment3M,CumulativeEnrollmentToDate,NextMonthEnrollment,NextMonthRevenue,CourseCategory,CourseType,CourseLevel,CoursePrice,CourseDuration,CourseRating
0,CR00001,2025-02,11,5195.08,18.0,8501.04,18.000000,18,8.0,3778.24,Programming,Paid,Beginner,472.28,11.0,4.74
1,CR00001,2025-03,8,3778.24,11.0,5195.08,14.500000,29,14.0,6611.92,Programming,Paid,Beginner,472.28,11.0,4.74
2,CR00001,2025-04,14,6611.92,8.0,3778.24,12.333333,37,17.0,8028.76,Programming,Paid,Beginner,472.28,11.0,4.74
3,CR00001,2025-05,17,8028.76,14.0,6611.92,11.000000,51,11.0,5195.08,Programming,Paid,Beginner,472.28,11.0,4.74
4,CR00001,2025-06,11,5195.08,17.0,8028.76,13.000000,68,12.0,5667.36,Programming,Paid,Beginner,472.28,11.0,4.74


## Phase 5: EDA

See `../reports/EDA_Report.md` for the full narrative and `../reports/figures/` for every chart. Key numeric findings reproduced here:

In [6]:
print('Enrollment stats:'); print(course_df['EnrollmentCount'].describe())
print('\nRevenue stats:'); print(course_df['TotalRevenue'].describe())
print('\nCorrelation (numeric features):')
num_cols = ['CoursePrice','CourseDuration','CourseRating','EnrollmentCount','TotalRevenue','AvgTeacherRating','AvgTeacherExperience']
course_df[num_cols].corr()

Enrollment stats:
count     60.000000
mean     166.666667
std       12.523424
min      140.000000
25%      157.500000
50%      166.000000
75%      174.750000
max      196.000000
Name: EnrollmentCount, dtype: float64

Revenue stats:
count       60.000000
mean     15188.724500
std      25409.721911
min          0.000000
25%          0.000000
50%          0.000000
75%      20858.060000
max      85416.600000
Name: TotalRevenue, dtype: float64

Correlation (numeric features):


,CoursePrice,CourseDuration,CourseRating,EnrollmentCount,TotalRevenue,AvgTeacherRating,AvgTeacherExperience
CoursePrice,1.000000,-0.096051,-0.031973,-0.163356,0.996722,-0.094174,0.258599
CourseDuration,-0.096051,1.000000,0.209411,-0.103414,-0.094198,-0.020527,-0.145062
CourseRating,-0.031973,0.209411,1.000000,0.294010,-0.018298,-0.013037,-0.069666
EnrollmentCount,-0.163356,-0.103414,0.294010,1.000000,-0.116611,0.018949,0.073369
TotalRevenue,0.996722,-0.094198,-0.018298,-0.116611,1.000000,-0.094761,0.253583
AvgTeacherRating,-0.094174,-0.020527,-0.013037,0.018949,-0.094761,1.000000,0.090688
AvgTeacherExperience,0.258599,-0.145062,-0.069666,0.073369,0.253583,0.090688,1.000000


## Phase 8-10: Train and evaluate models

In [7]:
from train_models import train_course_level_target, train_forecasting_target

enroll_results, enroll_pipes, enroll_feats = train_course_level_target(course_df, 'EnrollmentCount')
enroll_results

,Model,MAE,RMSE,R2
1,Ridge,10.440428,12.626868,-0.033819
0,LinearRegression,10.536716,13.111385,-0.114680
2,Lasso,10.542689,12.636705,-0.035430
3,RandomForest,10.547611,12.595669,-0.028716
4,GradientBoosting,11.152909,13.506732,-0.182915


In [8]:
revenue_results, revenue_pipes, revenue_feats = train_course_level_target(course_df, 'TotalRevenue')
revenue_results

,Model,MAE,RMSE,R2
1,Ridge,1713.902656,2566.989817,0.989621
2,Lasso,1924.843038,2553.587831,0.989729
0,LinearRegression,1946.376285,2576.227569,0.989546
3,RandomForest,2079.964420,4375.769086,0.969842
4,GradientBoosting,2148.643127,4229.816792,0.971820


In [9]:
fc_results, fc_pipes, fc_feats, split_info = train_forecasting_target(forecast_df)
print('train/test rows, test months:', split_info)
fc_results

train/test rows, test months: (480, 120, [Period('2025-10', 'M'), Period('2025-11', 'M')])


,Model,MAE,RMSE,R2
2,Lasso,2.758750,3.677855,-0.000371
1,Ridge,2.762627,3.767399,-0.049676
0,LinearRegression,2.766027,3.770292,-0.051288
3,RandomForest,3.366028,4.403678,-0.434178
4,GradientBoosting,3.661577,4.630833,-0.585953


## Phase 11: Feature importance (RandomForest, association not causation)

In [10]:
from evaluate_models import compute_and_plot_importance

enroll_imp = compute_and_plot_importance(course_df, 'EnrollmentCount', 'nb_enrollment_importance.png', 'Enrollment Feature Importance')
enroll_imp.head(10)

,feature,importance
2,CourseRating,0.233314
1,CourseDuration,0.208053
5,DistinctTeacherCount,0.108752
4,AvgTeacherExperience,0.095494
9,CourseCategory_Data Science,0.078436
3,AvgTeacherRating,0.066855
0,CoursePrice,0.047143
22,CourseLevel_Intermediate,0.036405
11,CourseCategory_Digital Marketing,0.029661
21,CourseLevel_Beginner,0.022277


In [11]:
revenue_imp = compute_and_plot_importance(course_df, 'TotalRevenue', 'nb_revenue_importance.png', 'Revenue Feature Importance')
revenue_imp.head(10)

,feature,importance
0,CoursePrice,0.971045
4,AvgTeacherExperience,0.006097
3,AvgTeacherRating,0.005171
5,DistinctTeacherCount,0.004544
2,CourseRating,0.003153
19,CourseType_Paid,0.002233
1,CourseDuration,0.001902
18,CourseType_Free,0.001740
6,CourseCategory_Artificial Intelligence,0.001030
7,CourseCategory_Business,0.000757


## Honest takeaway

Revenue models score R² ≈ 0.99 only because `Amount == CoursePrice` for every transaction in this dataset — revenue is a deterministic function of price × enrollment. Enrollment (true demand) models score R² ≈ 0 across every algorithm tried, including a chronologically-validated forecasting variant — the available catalog/instructor attributes do not carry a strong learnable demand signal in this dataset. See `../reports/Model_Evaluation.md` for full detail.